# Image-to-3D Unified Benchmark (FIXED)

**TripoSR vs LGM vs TRELLIS — same-protocol comparison on GSO.**

### What was fixed vs. the Sep 2026 run
1. **GPU hard-check** in cell 1 — the old run died silently on a CPU-only kernel; now you get a clear message instead.
2. **Protected installs** — no `pip install` line is allowed to replace Kaggle's CUDA torch build (checked after every install cell).
3. **Headless rendering fixed** — `PYOPENGL_PLATFORM=egl` is set *before* any OpenGL import, otherwise pyrender fails silently and PSNR/LPIPS silently vanish.
4. **TRELLIS on T4** — uses `xformers` attention backend instead of flash-attn (no 30-minute compile), 12 sampling steps.
5. **Isolated engine cells** — if TRELLIS crashes, TripoSR/LGM results are already saved to `/kaggle/working/outputs/`.
6. **GSO auto-detect + self-rendering** — if GT meshes exist but have no images, the notebook renders input + novel views itself with a camera identical to the evaluator.

### How to run
1. Kaggle → *Create → New Notebook* → **File → Import Notebook** → upload this file.
2. Right sidebar → **Session options → Accelerator → GPU T4 x2** and **Internet → ON**.
3. **Add Input → Datasets** → attach `anubhavaanand/image-to-3d-benchmark-data` (your benchmark code).
4. Run cells top to bottom. Start with `NUM_SAMPLES = 10` (smoke test), then set 100 for the paper run.


In [ ]:
import os, subprocess, sys, glob, gc, json, tarfile
from pathlib import Path

# --- EGL must be set before ANY OpenGL/pyrender import (headless GPU rendering) ---
os.environ["PYOPENGL_PLATFORM"] = "egl"
# --- TRELLIS backends that work on Kaggle without compiling flash-attn ---
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"

print("=== nvidia-smi ===")
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "!! nvidia-smi NOT FOUND - no GPU attached to this session")

import torch
print(f"torch {torch.__version__} (CUDA build: {torch.version.cuda})")
if not torch.cuda.is_available():
    raise RuntimeError(
        "torch.cuda.is_available() is False: this kernel is running on CPU.\n"
        "Fix: right sidebar > Session options > Accelerator > GPU T4 x2, then re-run all cells.\n"
        "(This is exactly what failed in the September 2026 run.)"
    )
props = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {props.name} | {props.total_memory/1e9:.1f} GB VRAM")

# ----------------- benchmark configuration -----------------
NUM_SAMPLES = 10   # smoke test first; set to 100 for the paper run
GSO_DIR = "/kaggle/working/data/gso"
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)
print(f"Config: NUM_SAMPLES={NUM_SAMPLES} | GSO_DIR={GSO_DIR}")


In [ ]:
# --- shared dependencies (none of these may replace torch) ---
!pip install -q trimesh scipy lpips rembg pyrender pandas "Pillow>=10.0"
import torch
assert torch.cuda.is_available(), "torch lost CUDA during installs - a package replaced the CUDA build"
print("shared deps installed, torch CUDA intact")


In [ ]:
# --- extract benchmark code from the attached Kaggle dataset ---
CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
if not os.path.exists(CODE_DIR):
    tars = glob.glob("/kaggle/input/image-to-3d-benchmark-data/*.tar.gz")
    assert tars, (
        "benchmark tarball not found - attach Kaggle dataset "
        "'anubhavaanand/image-to-3d-benchmark-data' via Add Input, then re-run"
    )
    with tarfile.open(tars[0]) as tf:
        tf.extractall("/kaggle/working")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.evaluation.benchmark_runner import BenchmarkRunner, load_gso_samples
from src.engines.triposr_engine import TriposrEngine
from src.engines.lgm_engine import LGMEngine
from src.engines.trellis_engine import TrellisEngine
print("benchmark code ready")


In [ ]:
# --- GSO data: attach / download from verified mirror / handpicked list / self-render ---
import numpy as np, trimesh, zipfile, tempfile, shutil, random
from PIL import Image as PILImage

def count_valid(root):
    root = Path(root)
    if not root.exists():
        return 0
    n = 0
    for d in root.iterdir():
        if d.is_dir() and (list((d/"meshes").glob("*.obj")) or list((d/"meshes").glob("*.ply"))):
            n += 1
    return n

# 1) auto-detect a GSO-like dataset already attached as a Kaggle input
if count_valid(GSO_DIR) == 0:
    for cand in sorted(glob.glob("/kaggle/input/*")):
        n = count_valid(cand)
        if n > 20:
            GSO_DIR = cand
            print(f"using attached dataset: {cand} ({n} objects)")
            break

# 2) ingest from the official raw GSO mirror (1,046 per-object archives)
#    Handpick workflow: create handpicked.txt (one object name per line, from
#    gso_catalog.txt) and upload it via Kaggle's 'Upload > File' - it appears
#    in /kaggle/working/ or under /kaggle/input/. Without it, a seeded random
#    sample (seed 42) is used.
REPO = "suvadityamuk/google-scanned-objects-raw"
if count_valid(GSO_DIR) == 0:
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi()
    all_zips = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.endswith(".zip"))
    print(f"mirror has {len(all_zips)} object archives")

    wanted, handpicked = None, False
    for hp in ["/kaggle/working/handpicked.txt"] + glob.glob("/kaggle/input/*/handpicked.txt"):
        if os.path.exists(hp):
            wanted = [l.strip() + ".zip" for l in open(hp)
                      if l.strip() and not l.strip().startswith("#")]
            wanted = [w for w in wanted if w in set(all_zips)]
            handpicked = True
            print(f"handpicked list: {len(wanted)} objects (from {hp})")
            break
    if wanted is None:
        random.Random(42).shuffle(all_zips)
        wanted = all_zips
        print(f"no handpicked.txt - using seeded random sample of {NUM_SAMPLES} (+spares)")

    def ingest(zip_name):
        obj_id = zip_name[:-4]
        try:
            zp = hf_hub_download(REPO, zip_name, repo_type="dataset",
                                 local_dir="/kaggle/working/gso_zips")
            with tempfile.TemporaryDirectory() as td:
                with zipfile.ZipFile(zp) as z:
                    z.extractall(td)
                hits = list(Path(td).rglob("*.obj")) + list(Path(td).rglob("*.ply"))
                if not hits:
                    return False
                mdir = Path(GSO_DIR) / obj_id / "meshes"
                mdir.mkdir(parents=True, exist_ok=True)
                for h in hits:
                    shutil.copy(h, mdir / h.name)
                for sib in hits[0].parent.iterdir():
                    if sib.suffix.lower() in (".mtl", ".png", ".jpg", ".jpeg"):
                        shutil.copy(sib, mdir / sib.name)
            return True
        except Exception as e:
            print("  skip", obj_id, str(e)[:80])
            return False

    target = len(wanted) if handpicked else NUM_SAMPLES + 10
    ok = 0
    for zname in wanted:
        if ingest(zname):
            ok += 1
        if ok >= target:
            break
    print(f"ingested {ok} objects into {GSO_DIR}")

def render_cam(tm, az, res, white_bg):
    import pyrender
    scene = pyrender.Scene()
    scene.add(pyrender.Mesh.from_trimesh(tm))
    if white_bg:
        scene.bg_color = np.array([1.0, 1.0, 1.0, 1.0])
    azr = np.radians(az)
    pose = np.array([
        [ np.cos(azr), 0, np.sin(azr), 1.5*np.sin(azr)],
        [ 0,           1, 0,           0             ],
        [-np.sin(azr), 0, np.cos(azr), 1.5*np.cos(azr)],
        [ 0,           0, 0,           1             ]])
    scene.add(pyrender.PerspectiveCamera(yfov=np.pi/3.0), pose=pose)
    scene.add(pyrender.DirectionalLight(color=np.ones(3), intensity=3.0))
    ren = pyrender.OffscreenRenderer(res, res)
    col, _ = ren.render(scene)
    ren.delete()
    return col

# 3) objects with meshes but no images: render input view (000, white bg, 512)
#    and GT novel view (001, black bg 256 - SAME camera as the evaluator)
missing = 0
for d in sorted(Path(GSO_DIR).iterdir()):
    if not d.is_dir():
        continue
    mcands = list((d/"meshes").glob("*.obj")) + list((d/"meshes").glob("*.ply"))
    if not mcands or (d/"images"/"000.png").exists():
        continue
    try:
        tm = trimesh.load(str(mcands[0]), force="mesh")
        if tm is None or len(tm.faces) == 0:
            continue
        try:
            tm.visual = tm.visual.to_color()   # bake UV textures to vertex colors
        except Exception:
            pass
        pts = np.array(tm.vertices, dtype=np.float32)
        c = (pts.max(0) + pts.min(0)) / 2
        s = np.linalg.norm(pts - c, axis=1).max()
        tm.apply_translation(-c); tm.apply_scale(1.0/(s+1e-8))
        (d/"images").mkdir(parents=True, exist_ok=True)
        PILImage.fromarray(render_cam(tm, 0.0, 512, True)).save(d/"images"/"000.png")
        PILImage.fromarray(render_cam(tm, 45.0, 256, False)).save(d/"images"/"001.png")
        missing += 1
    except Exception as e:
        print("render failed:", d.name, str(e)[:100])
print(f"self-rendered images for {missing} objects")

n = count_valid(GSO_DIR)
print(f"GSO ready: {n} objects at {GSO_DIR}")
if n == 0:
    print("NO GSO data found - check the messages above, or attach a dataset with "
          "<id>/meshes/*.obj layout via Add Input and re-run this cell.")
else:
    samples = load_gso_samples(GSO_DIR, num_samples=NUM_SAMPLES)
    print(f"runner loaded {len(samples)} samples")


In [ ]:
# --- TripoSR (triplane NeRF, ~1-2 s/object on T4) ---
!pip install -q git+https://github.com/VAST-AI-Research/TripoSR.git
!pip install -q torchmcubes xatlas==0.0.9 omegaconf einops
import torch
assert torch.cuda.is_available(), 'torch lost CUDA during installs'

tr_engine = TriposrEngine(device='cuda')
tr_engine.load_model()
runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[tr_engine], num_samples=NUM_SAMPLES, output_dir='/kaggle/working/outputs/triposr')
runner.run()
runner.save_results('/kaggle/working/outputs/triposr_results.json')
display(runner.get_summary_df())

del tr_engine, runner; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- LGM (multi-view Gaussians, ~10-20 s/object on T4) - NOTE: outputs Gaussians; geometry metrics need the optional mesh-conversion step ---
!pip install -q kiui tyro einops roma plyfile
!pip install -q git+https://github.com/ashawkey/mvdream.git
!git clone --depth 1 https://github.com/3DTopia/LGM.git /kaggle/working/LGM
!git clone --depth 1 https://github.com/ashawkey/diff-gaussian-rasterization.git /kaggle/working/dgr
!pip install -q /kaggle/working/dgr
import torch
assert torch.cuda.is_available(), 'torch lost CUDA during installs'

lg_engine = LGMEngine(device='cuda')
lg_engine.load_model()
runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[lg_engine], num_samples=NUM_SAMPLES, output_dir='/kaggle/working/outputs/lgm')
runner.run()
runner.save_results('/kaggle/working/outputs/lgm_results.json')
display(runner.get_summary_df())

del lg_engine, runner; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- TRELLIS (structured latents + flow, ~30-90 s/object on T4) - if OOM, re-create the engine with slat_steps=8, sparse_steps=8 ---
# spconv wheel must match Kaggle's CUDA version
import torch as _t
cu = (_t.version.cuda or '').replace('.', '')
ok = os.system(f'pip install -q spconv-cu{cu}') == 0
if not ok:
    for alt in ['spconv-cu124', 'spconv-cu121', 'spconv-cu120']:
        if os.system(f'pip install -q {alt}') == 0:
            ok = True; break
print('spconv installed:', ok)
# xformers: only install if missing, with --no-deps so torch is never replaced
try:
    import xformers
except ImportError:
    os.system('pip install -q --no-deps xformers')
!pip install -q pytorch-lightning omegaconf easydict 'imageio[ffmpeg]' timm kornia
!pip install -q git+https://github.com/NVlabs/nvdiffrast.git
!git clone --depth 1 https://github.com/microsoft/TRELLIS.git /kaggle/working/TRELLIS
if '/kaggle/working/TRELLIS' not in sys.path:
    sys.path.insert(0, '/kaggle/working/TRELLIS')
import torch
assert torch.cuda.is_available(), 'torch lost CUDA during installs'

tr_engine2 = TrellisEngine(device='cuda', output_format='mesh', slat_steps=12, sparse_steps=12)
tr_engine2.load_model()
runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[tr_engine2], num_samples=NUM_SAMPLES, output_dir='/kaggle/working/outputs/trellis')
runner.run()
runner.save_results('/kaggle/working/outputs/trellis_results.json')
display(runner.get_summary_df())

del tr_engine2, runner; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# --- merge results, summary table, LaTeX export, Pareto plot ---
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for p in sorted(glob.glob("/kaggle/working/outputs/*_results.json")):
    try:
        rows += json.load(open(p))
    except Exception as e:
        print("skip", p, str(e)[:80])

assert rows, "no results found - run at least one engine cell above"
df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/outputs/unified_results.csv", index=False)

num_cols = ["latency_s", "chamfer_distance", "f_score_0.1", "f_score_0.2",
            "f_score_0.5", "psnr", "lpips"]
cols = [c for c in num_cols if c in df.columns]
summary = df.groupby("engine")[cols].agg(["mean", "std"]).round(4)
print("=== UNIFIED BENCHMARK RESULTS (mean +/- std) ===")
print(summary.to_string())

latex = summary.reset_index().to_latex(index=False, na_rep="--")
with open("/kaggle/working/outputs/unified_table.tex", "w") as f:
    f.write(latex)
print("saved: unified_results.csv, unified_table.tex")

colors = {"TripoSR": "#1f77b4", "LGM": "#ff7f0e", "TRELLIS": "#2ca02c"}
means = df.groupby("engine")[cols].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for eng, row in means.iterrows():
    c = colors.get(eng, "grey")
    if "f_score_0.1" in row and not pd.isna(row.get("f_score_0.1")):
        axes[0].scatter(row["latency_s"], row["f_score_0.1"], s=180, c=c, label=eng, zorder=5)
    if "chamfer_distance" in row and not pd.isna(row.get("chamfer_distance")):
        axes[1].scatter(row["latency_s"], row["chamfer_distance"], s=180, c=c, label=eng, zorder=5)
axes[0].set_xlabel("Latency (s)"); axes[0].set_ylabel("F-Score @ 0.1 (higher better)")
axes[0].set_title("Speed vs F-Score"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("Latency (s)"); axes[1].set_ylabel("Chamfer Distance (lower better)")
axes[1].set_title("Speed vs Chamfer Distance"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/outputs/pareto_frontier.png", dpi=150)
plt.show()
print("saved: pareto_frontier.png - download /kaggle/working/outputs/ via the notebook Output tab")
